# Lab 09: Groups


**Helpful Resource:**
- [Python Reference](http://data8.org/sp24/reference/): Cheat sheet of helpful array & table methods!

**Recommended Readings**:

* [Visualizing Numerical Distributions](https://www.inferentialthinking.com/chapters/07/2/Visualizing_Numerical_Distributions.html)
* [Functions and Tables](https://www.inferentialthinking.com/chapters/08/Functions_and_Tables.html)

Please complete this notebook by filling in the cells provided. **Before you begin, execute the cell below to setup the notebook by importing some helpful libraries.** Each time you start your server, you will need to execute this cell again.


In [4]:
!apt-get install texlive texlive-xetex texlive-latex-extra pandoc
!pip install pypandoc
!pip install datascience

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
pandoc is already the newest version (2.9.2.1-3ubuntu2).
pandoc set to manually installed.
The following additional packages will be installed:
  dvisvgm fonts-droid-fallback fonts-lato fonts-lmodern fonts-noto-mono
  fonts-texgyre fonts-urw-base35 libapache-pom-java libcommons-logging-java
  libcommons-parent-java libfontbox-java libgs9 libgs9-common libidn12
  libijs-0.35 libjbig2dec0 libkpathsea6 libpdfbox-java libptexenc1 libruby3.0
  libsynctex2 libteckit0 libtexlua53 libtexluajit2 libwoff1 libzzip-0-13
  lmodern poppler-data preview-latex-style rake ruby ruby-net-telnet
  ruby-rubygems ruby-webrick ruby-xmlrpc ruby3.0 rubygems-integration t1utils
  teckit tex-common tex-gyre texlive-base texlive-binaries
  texlive-fonts-recommended texlive-latex-base texlive-latex-recommended
  texlive-pictures texlive-plain-generic tipa xfonts-encodings xfonts-utils
Suggested packages:
  fonts-noto f

In [5]:
# Connect Google Drive to Colab so you can access your files
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir('/content/drive/MyDrive/Colab Notebooks/')

Mounted at /content/drive


In [6]:
# Run this cell to set up the notebook, but please don't change it.

# These lines import the Numpy and Datascience modules.
import numpy as np
from datascience import *

# These lines do some fancy plotting magic.
import matplotlib
%matplotlib inline
import matplotlib.pyplot as plt
plt.style.use('fivethirtyeight')

import warnings
warnings.simplefilter('ignore', FutureWarning)
warnings.filterwarnings("ignore")

## 1. Based on the Lecture Activity

In [7]:
drinks = Table(['Drink', 'Cafe', 'Price'])
drinks = drinks.with_rows([
    ['Milk Tea', 'Asha', 5.5],
    ['Espresso', 'Strada',  1.75],
    ['Latte',    'Strada',  3.25],
    ['Espresso', "FSM",   2]
])
drinks

Drink,Cafe,Price
Milk Tea,Asha,5.5
Espresso,Strada,1.75
Latte,Strada,3.25
Espresso,FSM,2


In [8]:
discounts = Table().with_columns(
    'Coupon % off', make_array(10, 25, 5),
    'Location', make_array('Asha', 'Strada', 'Asha')
)
discounts

Coupon % off,Location
10,Asha
25,Strada
5,Asha


<br>


**Question 1** Join drinks and discounts based on Cafe/Location.

In [11]:
combined = drinks.join("Cafe",discounts,"Location")
combined

Cafe,Drink,Price,Coupon % off
Asha,Milk Tea,5.5,10
Asha,Milk Tea,5.5,5
Strada,Espresso,1.75,25
Strada,Latte,3.25,25


<br>


**Question 2** Calculate the lowest rate you can get for each drink using the available coupons.

In [61]:
discount_frac = combined.column("Coupon % off") / 100
combined.with_column("apply coupon",combined.column("Price") * (1 - discount_frac)).group("Drink", min)

Drink,Cafe min,Price min,Coupon % off min,apply coupon min
Espresso,Strada,1.75,25,1.3125
Latte,Strada,3.25,25,2.4375
Milk Tea,Asha,5.5,5,4.95


<br>

**Question 3** What happens if I run the following? How many rows will it produce? Why?


In [62]:
drinks.join('Cafe', drinks, 'Cafe')

Cafe,Drink,Price,Drink_2,Price_2
Asha,Milk Tea,5.5,Milk Tea,5.5
FSM,Espresso,2,Espresso,2
Strada,Espresso,1.75,Espresso,1.75
Strada,Espresso,1.75,Latte,3.25
Strada,Latte,3.25,Espresso,1.75
Strada,Latte,3.25,Latte,3.25


The code drinks.join('Cafe', drinks, 'Cafe') performs a self-join of the drinks table. In a join operation, when the join key matches, every possible pair of rows from the left table and the right table is combined.
Specifically, the original table has 'Strada' appearing twice, so joining on 'Cafe' will produce 2×2=4 rows for 'Strada'. Since 'Asha' and 'FSM' each appear once, they will generate 1×1=1 row each. Therefore, the resulting table will have a total of 4+1+1=6 rows.

## 2. Bike Sharing

In [26]:
trip = Table.read_table('./DS/trip.csv')
trip.show(3)

Trip ID,Duration,Start Date,Start Station,Start Terminal,End Date,End Station,End Terminal,Bike #,Subscriber Type,Zip Code
913460,765,8/31/2015 23:26,Harry Bridges Plaza (Ferry Building),50,8/31/2015 23:39,San Francisco Caltrain (Townsend at 4th),70,288,Subscriber,2139
913459,1036,8/31/2015 23:11,San Antonio Shopping Center,31,8/31/2015 23:28,Mountain View City Hall,27,35,Subscriber,95032
913455,307,8/31/2015 23:13,Post at Kearny,47,8/31/2015 23:18,2nd at South Park,64,468,Subscriber,94107


In [31]:
a=trip.column(3)
items=np.unique(a, return_counts=True)
for k,v in zip(items[0],items[1]):
    print(k,v)

2nd at Folsom 5396
2nd at South Park 6065
2nd at Townsend 9418
5th at Howard 5324
Adobe on Almaden 392
Arena Green / SAP Center 430
Beale at Market 5621
Broadway St at Battery St 5288
California Ave Caltrain Station 288
Castro Street and El Camino Real 828
Civic Center BART (7th at Market) 5367
Clay at Battery 3587
Commercial at Montgomery 4060
Cowper at University 403
Davis at Jackson 3639
Embarcadero at Bryant 5248
Embarcadero at Folsom 5331
Embarcadero at Sansome 10048
Embarcadero at Vallejo 3530
Evelyn Park and Ride 611
Franklin at Maple 54
Golden Gate at Polk 2545
Grant Avenue at Columbus Avenue 5667
Harry Bridges Plaza (Ferry Building) 12201
Howard at 2nd 4754
Japantown 633
MLK Library 798
Market at 10th 8006
Market at 4th 6527
Market at Sansome 7632
Mechanics Plaza (Market at Battery) 4269
Mezes Park 173
Mountain View Caltrain Station 2626
Mountain View City Hall 1135
Palo Alto Caltrain Station 815
Park at Olive 263
Paseo de San Antonio 582
Post at Kearny 3122
Powell Street BART

<br>

**Question 1** Create pivot table for Start Station and End Station. What value is being shown in each cell?

In [34]:
pivotTable = trip.pivot('Start Station', 'End Station')
pivotTable
#The value shown in each cell of this table represents the count of bike rentals for trips originating at the corresponding 'Start Station' and ending at the corresponding 'End Station'.
#In other words, it is the aggregated number of trips between a specific starting point and a specific destination.

End Station,2nd at Folsom,2nd at South Park,2nd at Townsend,5th at Howard,Adobe on Almaden,Arena Green / SAP Center,Beale at Market,Broadway St at Battery St,California Ave Caltrain Station,Castro Street and El Camino Real,Civic Center BART (7th at Market),Clay at Battery,Commercial at Montgomery,Cowper at University,Davis at Jackson,Embarcadero at Bryant,Embarcadero at Folsom,Embarcadero at Sansome,Embarcadero at Vallejo,Evelyn Park and Ride,Franklin at Maple,Golden Gate at Polk,Grant Avenue at Columbus Avenue,Harry Bridges Plaza (Ferry Building),Howard at 2nd,Japantown,MLK Library,Market at 10th,Market at 4th,Market at Sansome,Mechanics Plaza (Market at Battery),Mezes Park,Mountain View Caltrain Station,Mountain View City Hall,Palo Alto Caltrain Station,Park at Olive,Paseo de San Antonio,Post at Kearny,Powell Street BART,Powell at Post (Union Square),Redwood City Caltrain Station,Redwood City Medical Center,Redwood City Public Library,Rengstorff Avenue / California Street,Ryland Park,SJSU - San Salvador at 9th,SJSU 4th at San Carlos,San Antonio Caltrain Station,San Antonio Shopping Center,San Francisco Caltrain (Townsend at 4th),San Francisco Caltrain 2 (330 Townsend),San Francisco City Hall,San Jose City Hall,San Jose Civic Center,San Jose Diridon Caltrain Station,San Mateo County Center,San Pedro Square,San Salvador at 1st,Santa Clara County Civic Center,Santa Clara at Almaden,South Van Ness at Market,Spear at Folsom,St James Park,Stanford in Redwood City,Steuart at Market,Temporary Transbay Terminal (Howard at Beale),Townsend at 7th,University and Emerson,Washington at Kearny,Yerba Buena Center of the Arts (3rd @ Howard)
2nd at Folsom,53,127,396,82,0,0,27,8,0,0,31,45,40,0,8,24,27,36,13,0,0,4,17,372,41,0,0,129,84,180,26,0,0,0,0,0,0,39,47,46,0,0,0,0,0,0,0,0,0,474,285,18,0,0,0,0,0,0,0,0,32,37,0,0,30,207,297,0,6,20
2nd at South Park,149,107,39,140,0,0,136,53,0,0,43,53,111,0,19,31,131,67,72,0,0,34,43,428,337,0,0,46,103,978,70,0,0,0,0,0,0,194,47,76,0,0,0,0,0,0,0,0,0,367,226,30,0,0,0,0,0,0,0,0,49,84,0,0,266,213,92,0,52,109
2nd at Townsend,349,109,164,57,0,0,424,238,0,0,56,207,102,0,258,276,441,315,215,0,0,16,213,1837,185,0,0,83,176,569,129,0,0,0,0,0,0,85,71,97,0,0,0,0,0,0,0,0,0,613,207,16,0,0,0,0,0,0,0,0,32,323,0,0,1685,489,324,0,49,97
5th at Howard,80,126,63,136,0,0,43,103,0,0,167,47,71,0,29,47,31,125,41,0,0,64,48,288,353,0,0,251,128,121,55,0,0,0,0,0,0,78,142,112,0,0,0,0,0,0,0,0,0,432,1244,42,0,0,0,0,0,0,0,0,65,56,0,0,145,558,135,0,30,218
Adobe on Almaden,0,0,0,0,19,6,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,14,3,0,0,0,0,0,0,0,0,0,24,0,0,0,0,0,0,0,8,6,12,0,0,0,0,0,13,13,176,0,18,4,3,4,0,0,11,0,0,0,0,0,0,0
Arena Green / SAP Center,0,0,0,0,9,80,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,14,5,0,0,0,0,0,0,0,0,0,14,0,0,0,0,0,0,0,20,2,4,0,0,0,0,0,4,17,7,0,32,9,21,154,0,0,5,0,0,0,0,0,0,0
Beale at Market,71,50,116,48,0,0,69,462,0,0,159,60,63,0,167,114,27,401,109,0,0,43,286,58,46,0,0,181,124,115,21,0,0,0,0,0,0,33,161,133,0,0,0,0,0,0,0,0,0,423,191,15,0,0,0,0,0,0,0,0,168,60,0,0,12,112,15,0,43,21
Broadway St at Battery St,39,59,189,99,0,0,745,115,0,0,49,227,188,0,131,167,47,201,30,0,0,4,58,125,32,0,0,25,79,246,162,0,0,0,0,0,0,108,50,150,0,0,0,0,0,0,0,0,0,438,295,8,0,0,0,0,0,0,0,0,11,72,0,0,233,504,30,0,59,30
California Ave Caltrain Station,0,0,0,0,0,0,0,0,68,1,0,0,0,24,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,4,138,31,0,0,0,0,0,0,0,7,0,0,0,11,7,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,48,0,0
Castro Street and El Camino Real,0,0,0,0,0,0,0,0,0,66,0,0,0,0,0,0,0,0,0,9,0,0,0,0,0,0,0,0,0,0,0,0,636,34,0,1,0,0,0,0,0,0,0,10,0,0,0,3,16,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0


<br>

**Question 2** Create pivot table showing average duration of the trips between the  Start Station and End Station ?

In [36]:
averagePivot = trip.pivot('Start Station', 'End Station', values='Duration', collect=np.mean)
averagePivot

End Station,2nd at Folsom,2nd at South Park,2nd at Townsend,5th at Howard,Adobe on Almaden,Arena Green / SAP Center,Beale at Market,Broadway St at Battery St,California Ave Caltrain Station,Castro Street and El Camino Real,Civic Center BART (7th at Market),Clay at Battery,Commercial at Montgomery,Cowper at University,Davis at Jackson,Embarcadero at Bryant,Embarcadero at Folsom,Embarcadero at Sansome,Embarcadero at Vallejo,Evelyn Park and Ride,Franklin at Maple,Golden Gate at Polk,Grant Avenue at Columbus Avenue,Harry Bridges Plaza (Ferry Building),Howard at 2nd,Japantown,MLK Library,Market at 10th,Market at 4th,Market at Sansome,Mechanics Plaza (Market at Battery),Mezes Park,Mountain View Caltrain Station,Mountain View City Hall,Palo Alto Caltrain Station,Park at Olive,Paseo de San Antonio,Post at Kearny,Powell Street BART,Powell at Post (Union Square),Redwood City Caltrain Station,Redwood City Medical Center,Redwood City Public Library,Rengstorff Avenue / California Street,Ryland Park,SJSU - San Salvador at 9th,SJSU 4th at San Carlos,San Antonio Caltrain Station,San Antonio Shopping Center,San Francisco Caltrain (Townsend at 4th),San Francisco Caltrain 2 (330 Townsend),San Francisco City Hall,San Jose City Hall,San Jose Civic Center,San Jose Diridon Caltrain Station,San Mateo County Center,San Pedro Square,San Salvador at 1st,Santa Clara County Civic Center,Santa Clara at Almaden,South Van Ness at Market,Spear at Folsom,St James Park,Stanford in Redwood City,Steuart at Market,Temporary Transbay Terminal (Howard at Beale),Townsend at 7th,University and Emerson,Washington at Kearny,Yerba Buena Center of the Arts (3rd @ Howard)
2nd at Folsom,2678.47,405.961,337.677,463.354,0,0,432.815,881.75,0,0,744.774,493.844,482.75,0,865.875,636.042,573.852,2659.83,12791.4,0,0,789,2473.59,505.489,197.537,0,0,728.357,433.917,323.517,353.769,0,0,0,0,0,0,442.051,611.043,485.891,0,0,0,0,0,0,0,0,0,473.416,484.032,1096.17,0,0,0,0,0,0,0,0,919.562,355.865,0,0,1195.13,311.667,641.572,0,709,401.95
2nd at South Park,264.678,1432.28,280.641,535.379,0,0,527.485,685.679,0,0,776.14,617.226,611.694,0,553.158,329.097,385.29,1081.19,643.722,0,0,655.441,1096.02,876.544,219.881,0,0,787.826,631.087,301.815,379.857,0,0,0,0,0,0,292.423,915.745,607.118,0,0,0,0,0,0,0,0,0,285.659,540.549,883.6,0,0,0,0,0,0,0,0,777.347,355.274,0,0,559.526,322.615,466.054,0,703.346,384.367
2nd at Townsend,431.779,538.037,2510.52,601.789,0,0,741.637,784.836,0,0,764.018,787.333,852.676,0,776.155,430.822,540.664,1265.01,1847.95,0,0,992.125,926.61,664.486,560.784,0,0,979.964,1227.18,516.301,944.14,0,0,0,0,0,0,1054.07,2778.37,733.557,0,0,0,0,0,0,0,0,0,354.341,317.546,1165.62,0,0,0,0,0,0,0,0,1115.56,430.18,0,0,559.707,535.278,567.623,0,1146.53,603.99
5th at Howard,499.175,681.151,3167.84,6305.24,0,0,932.884,891.777,0,0,466.036,705.979,649.38,0,1020,636.532,606.194,1315.94,3204.76,0,0,481.438,832.542,727.91,425.048,0,0,735.884,343.273,410.421,634.945,0,0,0,0,0,0,551.744,913.725,410.955,0,0,0,0,0,0,0,0,0,428.13,401.162,686.905,0,0,0,0,0,0,0,0,550.785,593.036,0,0,836.117,405.131,576.815,0,1024.3,241.362
Adobe on Almaden,0,0,0,0,2865.63,3475.33,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,930.429,4636,0,0,0,0,0,0,0,0,0,893.167,0,0,0,0,0,0,0,3035.5,871.167,503.083,0,0,0,0,0,558.538,329.154,1012.24,0,369.611,684,1063,403.25,0,0,461.364,0,0,0,0,0,0,0
Arena Green / SAP Center,0,0,0,0,1506.11,4942.16,0,0,0,0,0,0,0,0,0,0,0,0,0,27409,0,0,0,0,0,1101.43,1921.6,0,0,0,0,0,0,0,0,0,737.857,0,0,0,0,0,0,0,683.25,916,788.25,0,0,0,0,0,547,798.353,983.286,0,354.688,1171.33,1345.33,247.688,0,0,745.8,0,0,0,0,0,0,0
Beale at Market,360.521,608.08,782.509,821.229,0,0,7148.71,639.268,0,0,618.767,518.85,319.143,0,224.174,447.825,331.593,595.736,391.78,0,0,662.953,448.297,3224.84,334.913,0,0,727.525,438.782,732.313,1302.71,0,0,0,0,0,0,2591.64,493.472,429.654,0,0,0,0,0,0,0,0,0,761.388,785.916,843.8,0,0,0,0,0,0,0,0,970.458,522.783,0,0,1404.5,562.866,1054.47,0,446.977,446.381
Broadway St at Battery St,712.513,914.102,761.27,1050.56,

## 3. San Francisco City Employee Salaries

This exercise is designed to give you practice with using the Table methods `.pivot` and `.group`. Here is a link to the [Python Reference](http://data8.org/sp24/reference/) in case you need a quick refresher. The [Table Function Visualizer](http://data8.org/interactive_table_functions/) may also be a helpful tool.


The data source we will use within this portion of the homework is [publicly provided](https://data.sfgov.org/City-Management-and-Ethics/Employee-Compensation/88g8-5mnd/data) by the City of San Francisco. We have filtered it to retain just the relevant columns and restricted the data to the calendar year 2019. Run the following cell to load our data into a table called `full_sf`.

In [37]:
full_sf = Table.read_table("./DS/sf2019.csv")
full_sf.show(5)

Organization Group,Department,Job Family,Job,Salary,Overtime,Benefits,Total Compensation
Public Protection,Adult Probation,Information Systems,IS Trainer-Journey,91332,0,40059,131391
Public Protection,Adult Probation,Information Systems,IS Engineer-Assistant,123241,0,49279,172520
Public Protection,Adult Probation,Information Systems,IS Business Analyst-Senior,115715,0,46752,162468
Public Protection,Adult Probation,Information Systems,IS Business Analyst-Principal,159394,0,57312,216706
Public Protection,Adult Probation,Information Systems,IS Programmer Analyst,70035,0,28671,98706


The table has one row for each of the 44,525 San Francisco government employees in 2019.

The first four columns describe the employee's job. For example, the employee in the third row of the table had a job called "IS Business Analyst-Senior". We will call this the employee's *position* or *job title*. The job was in a Job Family called Information Systems (hence the IS in the job title), and was in the Adult Probation Department that is part of the Public Protection Organization Group of the government. You will mostly be working with the `Job` column.

The next three columns contain the dollar amounts paid to the employee in the calendar year 2019 for salary, overtime, and benefits. Note that an employee’s salary does not include their overtime earnings.

The last column contains the total compensation paid to the employee. It is the sum of the previous three columns:

$$\text{Total Compensation} = \text{Salary} + \text{Overtime} + \text{Benefits}$$

For this homework, we will be using the following columns:
1. `Organization Group`: A group of departments. For example, the Public Protection Org. Group includes departments such as the Police, Fire, Adult Protection, District Attorney, etc.
2. `Department`: The primary organizational unit used by the City and County of San Francisco.
3. `Job`: The specific position that a given worker fills.
4. `Total Compensation`: The sum of a worker's salary, overtime, and benefits in 2019.


Run the following cell to select the relevant columns and create a new table named `sf`.

In [38]:
sf = full_sf.select("Job", "Department", "Organization Group",  "Total Compensation")
sf.show(5)

Job,Department,Organization Group,Total Compensation
IS Trainer-Journey,Adult Probation,Public Protection,131391
IS Engineer-Assistant,Adult Probation,Public Protection,172520
IS Business Analyst-Senior,Adult Probation,Public Protection,162468
IS Business Analyst-Principal,Adult Probation,Public Protection,216706
IS Programmer Analyst,Adult Probation,Public Protection,98706


We want to use this table to generate arrays with the job titles of the members of each **Organization Group**.

**Question 1.** Set `job_titles` to a table with two columns. The first column should be called `Organization Group` and have the name of every "Organization Group" each listed only once in this column, and the second column should be called `Jobs` with each row in that second column containing an *array* of the names of all the job titles within that "Organization Group". Don't worry if there are multiple of the same job titles. **(9 Points)**

*Hint 1:* Think about how `group` works: it collects values into an array and then applies a function to that array. We have defined two functions below for you, and you will need to use one of them in your call to `group`.

*Hint 2:* You might need to rename one of the columns.


In [46]:
# Pick one of the two functions defined below in your call to group.
def first_item(array):
    '''Returns the first item'''
    return array.item(0)

def full_array(array):
    '''Returns the array that is passed through'''
    return array

# Make a call to group using one of the functions above when you define job_titles
job_titles = sf.group("Organization Group", collect=full_array).relabeled("Job full_array", "Jobs").select("Organization Group","Jobs")
job_titles

Organization Group,Jobs
Community Health,"['Painter Supervisor 1' 'Painter' 'Painter' ..., 'Nursin ..."
Culture & Recreation,['Electrician' 'Executive Secretary 2' 'Bldgs & Grounds ...
General Administration & Finance,"['Painter' 'Painter' 'Electrician' ..., 'Investigator, T ..."
Human Welfare & Neighborhood Development,['Dept Head I' 'Administrative Analyst' 'Community Devel ...
Public Protection,['IS Trainer-Journey' 'IS Engineer-Assistant' 'IS Busine ...
"Public Works, Transportation & Commerce",['Heavy Equip Ops Asst Sprv' 'Heavy Equipment Ops Sprv' ...


<!-- BEGIN QUESTION -->

**Question 2.** At the moment, the `Job` column of the `sf` table is not sorted (no particular order). Would the arrays you generated in the `Jobs` column of the previous question be the same if we had sorted alphabetically instead before generating them? Explain your answer. To receive full credit, your answer should reference *how* the `.group` method works, and how sorting the `Jobs` column would affect this.  **(8 Points)**

*Note:* Two arrays are the **same** if they contain the same number of elements and the elements located at corresponding indexes in the two arrays are identical. An example of arrays that are NOT the same: `array([1,2]) != array([2,1])`.


The Table.group() method preserves the original row order when collecting values within a group. Therefore, if the sf table were sorted beforehand by the 'Job' column, the order in which .group places the job titles into the resulting arrays would change. Since the arrays must match not only in content but also in the order of their elements to be considered the same, the arrays generated before and after sorting would be different.

<!-- END QUESTION -->

**Question 3.** Set `department_ranges` to a table containing departments as the rows, and the organization groups as the columns. The values in the rows should correspond to a total compensation range, where range is defined as the **difference between the highest total compensation and the lowest total compensation in the department for that organization group**. **(9 Points)**

*Hint:* First you'll need to define a new function `compensation_range` which takes in an array of compensations and returns the range of compensations in that array.


In [51]:
# Define compensation_range first
def compensation_range(array):
    '''Returns the range of compensations in the array'''
    return max(array) - min(array)

department_ranges = sf.pivot("Department", "Organization Group", values="Total Compensation", collect=compensation_range)
department_ranges

Organization Group,Academy Of Sciences,Administrative Services,Adult Probation,Airport Commission,Art Commission,Asian Art Museum,Assessor,Board Of Appeals,Board Of Supervisors,Building Inspection,Child Support Services,Children & Families Commission,Children Youth & Families,City Attorney,City Planning,Civil Service Commission,Controller,Department Of Public Works,Department of Technology,Dept Status of Women,Dept of Emergency Management,Dept of Police Accountablility,District Attorney,Economic Workforce Development,Environment,Ethics Commission,Fine Arts Museum,Fire Department,Health Service System,Homeless Services,Human Resources,Human Rights Commission,Human Services,Juvenile Court,Law Library,Mayor,Municipal Transportation Agcy,Police,Port,Public Defender,Public Health,Public Library,Public Utilities Commission,Recreation And Park Commission,Registrar,Rent Arbitration Board,Retirement Services,Sheriff,Treasurer/Tax Collector,Trial Courts,War Memorial
Community Health,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,554179,0,0,0,0,0,0,0,0,0,0
Culture & Recreation,199121,0,0,0,251823,298230,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,286064,0,0,0,0,0,0,0,56713,0,0,0,0,0,0,322249,0,341046,0,0,0,0,0,0,219833
General Administration & Finance,0,478784,0,0,0,0,277385,0,293773,0,0,0,0,419920,359011,202506,409401,0,349757,0,0,0,0,0,0,259960,0,0,331576,0,335840,0,0,0,0,440980,0,0,0,0,0,0,0,0,297729,0,723774,0,332980,0,0
Human Welfare & Neighborhood Development,0,0,0,0,0,0,0,0,0,0,313210,56318,294674,0,0,0,0,0,0,231712,0,0,0,0,290205,0,0,0,0,316098,0,301206,418740,0,0,0,0,0,0,0,0,0,0,0,0,293132,0,0,0,0,0
Public Protection,0,0,303419,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,325333,332041,335621,0,0,0,0,492128,0,0,0,0,0,342453,0,0,0,440552,0,337095,0,0,0,0,0,0,0,545324,0,320807,0
"Public Works, Transportation & Commerce",0,0,0,445092,0,0,0,243582,0,340852,0,0,0,0,0,0,0,374263,0,0,0,0,0,300888,0,0,0,0,0,0,0,0,0,0,0,0,381639,0,393270,0,0,0,489773,0,0,0,0,0,0,0,0


In [50]:
# Define compensation_range first
def compensation_range(array):
    '''Returns the range of compensations in the array'''
    return max(array) - min(array)

<!-- BEGIN QUESTION -->

**Question 4.** Why might some of the row values be `0` in the `department_ranges` table from the previous question. **(8 Points)**


The values of 0 in the department_ranges table occur because the Total Compensation values for all employees within that specific combination of Organization Group and Department are identical. The compensation_range function calculates the difference between the maximum and minimum values in an array (max(array)−min(array)). Therefore, a return value of 0 indicates that the range of salaries for that particular group is 0, meaning there is no variation in the total compensation paid.

<!-- END QUESTION -->

**Question 5.** Find the number of departments appearing in the `sf` table that have an average total compensation of greater than 125,000 dollars; assign this value to the variable `num_over_125k`. **(9 Points)**

*Note:* The variable names provided are meant to help guide the intermediate steps and general thought process. Feel free to delete them if you'd prefer to start from scratch, but make sure your final answer is assigned to `num_over_125k`!


In [56]:
depts_and_comp = sf.select("Department", "Total Compensation")
department_avg = depts_and_comp.group("Department", np.mean)
num_over_125k = department_avg.where("Total Compensation mean", are.above(125000)).num_rows
num_over_125k

23

## Submission

Make sure you have run all cells in your notebook in order before running the cell below, so that all images/graphs appear in the output. The cell below will generate a pdf file for you to submit. **Please save before exporting!**

In [ ]:
# should change the directory and file name matching to yours
!jupyter nbconvert './DS/lab09(SSU).ipynb' --to pdf

[NbConvertApp] Converting notebook ./DS/lab09(SSU).ipynb to pdf
[NbConvertApp] Writing 96912 bytes to notebook.tex
[NbConvertApp] Building PDF
[NbConvertApp] Running xelatex 3 times: ['xelatex', 'notebook.tex', '-quiet']
